In [10]:
import math

# --- Value Class (Micro Autograd Engine) ---
class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    # ReLU activation
    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), 'ReLU')

        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad

        out._backward = _backward
        return out

    def backward(self):

        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


# --- Micro Autograd Neuron Example ---
print("--- Micro-Autograd Neuron ---")

x1 = Value(2.0)
x2 = Value(-3.0)
w1 = Value(-1.0)
w2 = Value(1.0)
b = Value(6.0)

# Forward pass
w1x1 = w1 * x1
w2x2 = w2 * x2
sum_val = w1x1 + w2x2
neuron = sum_val + b

out = neuron.relu()

print("Forward Pass Output:", out.data)

# Backward pass
out.backward()

print("\n--- Gradients ---")
print("Gradient w1:", w1.grad)
print("Gradient w2:", w2.grad)
print("Gradient x1:", x1.grad)
print("Gradient x2:", x2.grad)
print("Gradient b :", b.grad)

--- Micro-Autograd Neuron ---
Forward Pass Output: 1.0

--- Gradients ---
Gradient w1: 2.0
Gradient w2: -3.0
Gradient x1: -1.0
Gradient x2: 1.0
Gradient b : 1.0
